# Notebook 82: Mask-Profile Transport and A3 Robustness Experiment

**Runtime:** CPU. A high-memory runtime is preferable.

## Scientific question

Does the page-clustered print-safety benefit of **A3_PIXEL_GATE** over **A0_TELEA** survive when paired clean/annotated seed masks are progressively broadened toward the morphology of reviewed real-world HITL masks?

## Direct answer to “is 3 pixels sufficient?”

That cannot be assumed. A 3-pixel dilation is sufficient only if it reaches or brackets the real HITL mask profile on the predeclared morphology statistics. This notebook therefore tests:

- `M0`: original paired seed;
- `M1` to `M5`: seed dilated by 1 to 5 pixels;
- `MT`: per-page target-matched dilation selected **without clean truth**, using the pooled HITL thickness profile and the smallest radius in `0..8` that best matches it.

The broader `0..5` fixed grid prevents an arbitrary conclusion at 3 pixels. The `MT` arm answers whether any fixed radius is an adequate proxy for production masks.

## Experimental controls

- A0 and A3 definitions remain unchanged.
- Every transformed seed still passes through the canonical frozen `prepare_mask(seed, 2)` correction contract.
- Clean truth is unavailable until every observable arm output is written and hashed.
- Pages with zero seed or zero AMBIGUOUS pixels are recorded and skipped for the A0-versus-A3 treatment comparison.
- Pixels outside each arm's transformed authoritative mask remain byte-for-byte observed.
- Statistical uncertainty is clustered by page, not pixel.
- This is a development transport experiment, not production authorisation.

The project separates annotation confidence from correction safety, uses reviewed masks as the intervention boundary, applies canonical dilation 2, and recognises mask depth/component geometry as relevant safety evidence. The HITL detector code supplied with this commission is used only to document the production source of draft masks; Notebook 82 operates on reviewed corrected HITL masks where available.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy pyarrow matplotlib scipy scikit-image


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, shutil, gc, math, sys, platform
import cv2, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
from skimage.morphology import skeletonize

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
V89_OPERATOR_PATH=ROOT/'v8_9_release'/'v8_9_operator.py'
P_PRINT_BUNDLE=ROOT/'observed_print_full_development_freeze_v8_2f'/'20261117_120424_v8_2f_full_development_p_print_freeze'/'p_print_provider'
REMOVE_BUNDLE=ROOT/'m5_remove_full_development_freeze_v8_8f'/'20261117_110512_notebook52f_v8_8_full_development_remove_freeze'/'full_development_remove'
NB72_ROOT=ROOT/'notebook72_frozen_v8_8_annotated_only_operational_trial'
CLEAN_DIR=ROOT/'source'/'clean'
ALIGNED_DIR=ROOT/'aligned'
PAIRED_SEED_DIR=ROOT/'masks'/'corrected'
MANIFEST_PATH=ROOT/'development_score_manifest.csv'

# Reviewed corrected HITL masks are authoritative for the real-world morphology audit.
HITL_REVIEWED_MASK_CANDIDATES=[
    ROOT/'human_in_the_middle'/'corrected_masks',
    ROOT/'human_in_the_middle'/'reviewed_masks',
    ROOT/'production_pilot'/'reviewed_masks',
]

# Draft masks are reported separately, never silently substituted for reviewed masks.
HITL_DRAFT_MASK_DIR=ROOT/'human_in_the_middle'/'draft_masks'
HITL_AUDIT_DIR=ROOT/'human_in_the_middle'/'audit'

FIXED_RADII=list(range(0,6))           # M0..M5
TARGET_SEARCH_RADII=list(range(0,9))   # candidate radii for MT
CANONICAL_PREP_DILATION=2
PPRINT_GATE=0.50
DARK_THRESHOLD=200
IMPROVEMENT_MARGIN=5
BOOTSTRAP_RESAMPLES=10000
SEED=42
EXPECTED_HITL=151

RUN=ROOT/'notebook82_mask_profile_transport'/(datetime.now().strftime('%Y%m%d_%H%M%S')+'_notebook82')
for p in [RUN,RUN/'inputs',RUN/'observable',RUN/'tables',RUN/'visual',RUN/'audit',RUN/'upload']:
    p.mkdir(parents=True,exist_ok=False)

ARMS=['A0_TELEA','A3_PIXEL_GATE']
MASK_ARMS=[f'M{r}' for r in FIXED_RADII]+['MT']

def require(x,msg):
    if not bool(x): raise RuntimeError(msg)

def sha256_file(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()

def array_hash(a):
    a=np.ascontiguousarray(a);h=hashlib.sha256();h.update(str(a.dtype).encode());h.update(str(a.shape).encode());h.update(a.tobytes());return h.hexdigest()

def latest(root,required):
    hits=[]
    if Path(root).exists():
        for p in Path(root).rglob('*'):
            if p.is_dir() and all((p/r).exists() for r in required): hits.append(p)
    return max(hits,key=lambda p:p.stat().st_mtime) if hits else None

def image_map(folder):
    folder=Path(folder)
    return {p.stem:p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in {'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}} if folder.exists() else {}

def read_gray(p):
    im=cv2.imread(str(p),cv2.IMREAD_GRAYSCALE)
    require(im is not None,f'Unreadable image: {p}')
    return im

def binary(a): return np.asarray(a)>0

def dilate_seed(seed,r):
    b=binary(seed).astype(np.uint8)
    if r<=0:return b>0
    k=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(2*r+1,2*r+1))
    return cv2.dilate(b,k)>0

def printed_ink(clean):
    return cv2.adaptiveThreshold(clean,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0

print('Run:',RUN)


## 1. Resolve the frozen V8.9 operator and input cohorts

The paired cohort requires clean, aligned annotated and reviewed corrected seed files with matching stems. The HITL mask cohort is resolved from authoritative review outputs. If no reviewed HITL directory is available, the notebook stops before claiming a production-profile comparison; draft masks are still inventoried for diagnosis.


In [ ]:
import importlib.util
require(V89_OPERATOR_PATH.exists(),f'Missing {V89_OPERATOR_PATH}')
spec=importlib.util.spec_from_file_location('v8_9_operator',V89_OPERATOR_PATH)
mod=importlib.util.module_from_spec(spec);sys.modules['v8_9_operator']=mod;spec.loader.exec_module(mod)
op=mod.load_v89_operator(P_PRINT_BUNDLE,REMOVE_BUNDLE)
require(type(op).__name__=='V89Operator',f'Expected V89Operator, got {type(op).__name__}')

clean_map=image_map(CLEAN_DIR);obs_map=image_map(ALIGNED_DIR);paired_seed_map=image_map(PAIRED_SEED_DIR)
manifest=pd.read_csv(MANIFEST_PATH,dtype=str).fillna('')
require({'page','source_group'}.issubset(manifest.columns),'Development manifest missing page/source_group')
meta=manifest.set_index('page')
paired_pages=sorted(set(clean_map)&set(obs_map)&set(paired_seed_map)&set(meta.index.astype(str)))
require(paired_pages,'No paired clean/annotated/seed pages found.')

reviewed_dir=None;hitl_reviewed={}
for candidate in HITL_REVIEWED_MASK_CANDIDATES:
    q=image_map(candidate)
    if len(q)>len(hitl_reviewed): reviewed_dir=candidate;hitl_reviewed=q
hitl_drafts=image_map(HITL_DRAFT_MASK_DIR)

cohort_inventory={
    'paired_pages':len(paired_pages),
    'paired_seed_dir':str(PAIRED_SEED_DIR),
    'hitl_reviewed_mask_dir':str(reviewed_dir) if reviewed_dir else None,
    'hitl_reviewed_masks':len(hitl_reviewed),
    'hitl_draft_masks':len(hitl_drafts),
    'hitl_draft_mask_dir':str(HITL_DRAFT_MASK_DIR),
}
(RUN/'inputs'/'cohort_inventory.json').write_text(json.dumps(cohort_inventory,indent=2))
print(json.dumps(cohort_inventory,indent=2))
require(len(hitl_reviewed)>0,'No reviewed HITL corrected-mask directory found. Set HITL_REVIEWED_MASK_CANDIDATES to the authoritative reviewed mask location; draft masks are not accepted as reviewed production morphology.')


## 2. Mask morphology functions

Thickness is not represented by raw area alone. The audit therefore records normalised coverage, skeleton-based width, distance-transform depth, perimeter/area, component fill, holes and component mergers under canonical dilation 2.


In [ ]:
def mask_metrics(mask,page_key,cohort,variant='ORIGINAL'):
    m=binary(mask);h,w=m.shape;area=int(m.sum())
    row={'page_key':str(page_key),'cohort':str(cohort),'variant':str(variant),'height':h,'width':w,'page_pixels':h*w,'seed_pixels':area,'seed_fraction':area/max(h*w,1)}
    if area==0:
        row.update({'components':0,'median_component_area':np.nan,'p90_component_area':np.nan,'mean_fill_ratio':np.nan,'skeleton_pixels':0,'area_per_skeleton_pixel':np.nan,'depth_mean':np.nan,'depth_median':np.nan,'depth_p90':np.nan,'depth_max':np.nan,'perimeter_pixels':0,'perimeter_area_ratio':np.nan,'holes':0,'prepared_pixels':0,'prepared_seed_expansion':np.nan,'prepared_components':0,'component_mergers_at_prepare':0})
        return row
    n,lab,stats,_=cv2.connectedComponentsWithStats(m.astype(np.uint8),8)
    areas=[];fills=[]
    for cid in range(1,n):
        a=int(stats[cid,cv2.CC_STAT_AREA]);bw=int(stats[cid,cv2.CC_STAT_WIDTH]);bh=int(stats[cid,cv2.CC_STAT_HEIGHT]);areas.append(a);fills.append(a/max(bw*bh,1))
    sk=skeletonize(m);depth=cv2.distanceTransform(m.astype(np.uint8),cv2.DIST_L2,5);dv=depth[m]
    contours,hier=cv2.findContours(m.astype(np.uint8),cv2.RETR_CCOMP,cv2.CHAIN_APPROX_NONE)
    perimeter=float(sum(cv2.arcLength(c,True) for c in contours));holes=0
    if hier is not None:holes=int(np.sum(hier[0,:,3]>=0))
    prepared=dilate_seed(m,CANONICAL_PREP_DILATION);pn,_,_,_=cv2.connectedComponentsWithStats(prepared.astype(np.uint8),8)
    row.update({
        'components':n-1,'median_component_area':float(np.median(areas)),'p90_component_area':float(np.quantile(areas,.9)),'mean_fill_ratio':float(np.mean(fills)),
        'skeleton_pixels':int(sk.sum()),'area_per_skeleton_pixel':float(area/max(int(sk.sum()),1)),
        'depth_mean':float(dv.mean()),'depth_median':float(np.median(dv)),'depth_p90':float(np.quantile(dv,.9)),'depth_max':float(dv.max()),
        'perimeter_pixels':perimeter,'perimeter_area_ratio':float(perimeter/area),'holes':holes,
        'prepared_pixels':int(prepared.sum()),'prepared_seed_expansion':float(prepared.sum()/area),'prepared_components':pn-1,
        'component_mergers_at_prepare':int(max((n-1)-(pn-1),0)),
    })
    return row

morph=[]
for p in paired_pages:morph.append(mask_metrics(read_gray(paired_seed_map[p]),p,'PAIRED','ORIGINAL'))
for p,path in sorted(hitl_reviewed.items()):morph.append(mask_metrics(read_gray(path),p,'HITL_REVIEWED','ORIGINAL'))
for p,path in sorted(hitl_drafts.items()):morph.append(mask_metrics(read_gray(path),p,'HITL_DRAFT','ORIGINAL'))
morph=pd.DataFrame(morph);morph.to_csv(RUN/'tables'/'mask_morphology_by_page.csv',index=False)
display(morph.groupby('cohort')[['seed_fraction','area_per_skeleton_pixel','depth_p90','depth_max','prepared_seed_expansion','component_mergers_at_prepare']].median())


## 3. Quantify the paired-versus-HITL morphology shift

The target profile uses reviewed HITL masks only. The comparison reports medians, quantiles, Mann-Whitney tests and Cliff's delta. Statistical significance is secondary to effect size and range overlap.


In [ ]:
MORPH_FEATURES=['seed_fraction','area_per_skeleton_pixel','depth_mean','depth_median','depth_p90','depth_max','perimeter_area_ratio','mean_fill_ratio','prepared_seed_expansion','component_mergers_at_prepare']

def cliffs_delta(x,y):
    x=np.asarray(pd.Series(x).dropna(),float);y=np.asarray(pd.Series(y).dropna(),float)
    if not len(x) or not len(y):return np.nan
    gt=sum(np.sum(a>y) for a in x);lt=sum(np.sum(a<y) for a in x)
    return float((gt-lt)/(len(x)*len(y)))

paired_m=morph[morph.cohort=='PAIRED'];hitl_m=morph[morph.cohort=='HITL_REVIEWED']
shift=[]
for f in MORPH_FEATURES:
    x=paired_m[f].dropna();y=hitl_m[f].dropna()
    stat,p=mannwhitneyu(x,y,alternative='two-sided') if len(x) and len(y) else (np.nan,np.nan)
    shift.append({'feature':f,'paired_n':len(x),'hitl_n':len(y),'paired_median':x.median(),'hitl_median':y.median(),'paired_p10':x.quantile(.1),'paired_p90':x.quantile(.9),'hitl_p10':y.quantile(.1),'hitl_p90':y.quantile(.9),'median_ratio_hitl_over_paired':y.median()/x.median() if len(x) and x.median()!=0 else np.nan,'cliffs_delta_hitl_vs_paired':cliffs_delta(y,x),'mann_whitney_p':p})
shift=pd.DataFrame(shift);shift.to_csv(RUN/'tables'/'paired_vs_hitl_morphology_shift.csv',index=False);display(shift)

TARGET_FEATURES=['area_per_skeleton_pixel','depth_p90','prepared_seed_expansion']
HITL_TARGET={f:float(hitl_m[f].median()) for f in TARGET_FEATURES}
HITL_SCALE={f:max(float(hitl_m[f].quantile(.75)-hitl_m[f].quantile(.25)),1e-6) for f in TARGET_FEATURES}
(RUN/'inputs'/'hitl_target_profile.json').write_text(json.dumps({'target_median':HITL_TARGET,'hitl_iqr_scale':HITL_SCALE,'features':TARGET_FEATURES},indent=2))
print('HITL target:',HITL_TARGET)


## 4. Select the truth-free target-matched radius

For each paired page, `MT` is the radius in `0..8` with minimum robust standardised distance to the pooled reviewed-HITL median profile. Clean truth is not loaded or consulted.


In [ ]:
def target_distance(metrics):
    return float(np.sqrt(np.mean([((metrics[f]-HITL_TARGET[f])/HITL_SCALE[f])**2 for f in TARGET_FEATURES])))

target_rows=[];transformed_morph=[]
for page in paired_pages:
    seed=read_gray(paired_seed_map[page])
    candidates=[]
    for r in TARGET_SEARCH_RADII:
        m=dilate_seed(seed,r);q=mask_metrics(m,page,'PAIRED',f'M{r}');q['radius']=r;q['target_distance']=target_distance(q);candidates.append(q);transformed_morph.append(q)
    best=min(candidates,key=lambda q:(q['target_distance'],q['radius']))
    target_rows.append({'page':page,'target_radius':int(best['radius']),'target_distance':float(best['target_distance']),**{f'MT_{f}':best[f] for f in TARGET_FEATURES}})
target=pd.DataFrame(target_rows);target.to_csv(RUN/'tables'/'target_matched_radius_by_page.csv',index=False)
pd.DataFrame(transformed_morph).to_csv(RUN/'tables'/'paired_radius_morphology_grid.csv',index=False)
print(target.target_radius.value_counts().sort_index())
print('Median target radius:',target.target_radius.median(),'P90:',target.target_radius.quantile(.9),'Max:',target.target_radius.max())


## 5. Define A0 and A3 under each transformed seed

The only treatment difference is AMBIGUOUS handling:

- A0: AMBIGUOUS → Telea.
- A3: preserve observed where calibrated `p_print >= 0.50`; otherwise Telea.

REMOVE, PRESERVE, V8.9 Ring 2 and outside-mask immutability remain identical within a mask-profile arm.


In [ ]:
def build_a0_a3(observed,transformed_seed,page):
    seed=(binary(transformed_seed).astype(np.uint8)*255)
    v89=op.run_page(observed,seed,page=page)
    mask=mod.prepare_mask(seed,CANONICAL_PREP_DILATION)
    telea=mod.telea_reconstruct(observed,mask)
    _,routes=mod.geometry_t80(observed,mask,telea)
    preserve=np.asarray(routes['preserve'],bool);remove=np.asarray(routes['remove'],bool);amb=np.asarray(routes['ambiguous'],bool);paper=np.asarray(routes['paper']);ring2=np.asarray(v89.ring2_mask,bool);pp=np.asarray(v89.p_print,np.float32)
    pp0=np.where(np.isfinite(pp),np.clip(pp,0,1),0)
    base=observed.copy();base[remove]=paper[remove];base[preserve]=observed[preserve]
    a0=base.copy();a0[amb]=telea[amb]
    a3=a0.copy();gate=amb&(pp0>=PPRINT_GATE);a3[gate]=observed[gate]
    for img in [a0,a3]:
        img[ring2]=observed[ring2]
        require(np.array_equal(img[mask==0],observed[mask==0]),f'{page}: outside-mask drift')
    return {'arms':{'A0_TELEA':a0,'A3_PIXEL_GATE':a3},'mask':mask>0,'preserve':preserve,'remove':remove,'ambiguous':amb,'ring2':ring2,'p_print':pp,'gate':gate}


## 6. Generate and lock every observable output

This cell is resumable. A page/profile directory is reused only when its complete output set exists. Partial directories fail closed. Zero-seed and zero-AMBIGUOUS arms are explicitly recorded as zero experimental exposure.


In [ ]:
def complete_dir(d):
    req=['seed.png','authority.png','ambiguous.png','ring2.png','p_print.npy','A0_TELEA.png','A3_PIXEL_GATE.png','metadata.json']
    return d.is_dir() and all((d/x).exists() for x in req)

def transformed_seed_for(page,seed,mask_arm):
    if mask_arm=='MT':
        r=int(target.set_index('page').loc[page,'target_radius'])
    else:r=int(mask_arm[1:])
    return dilate_seed(seed,r),r

observable_rows=[]
for page in paired_pages:
    obs=read_gray(obs_map[page]);seed=read_gray(paired_seed_map[page]);require(obs.shape==seed.shape,f'{page}: shape mismatch')
    for mask_arm in MASK_ARMS:
        t_seed,radius=transformed_seed_for(page,seed,mask_arm);d=RUN/'observable'/page/mask_arm
        if complete_dir(d):
            meta_j=json.loads((d/'metadata.json').read_text());observable_rows.append(meta_j);print(page,mask_arm,'REUSE');continue
        if d.exists():raise RuntimeError(f'{page} {mask_arm}: partial directory {d}')
        if not t_seed.any():
            observable_rows.append({'page':page,'source_group':str(meta.loc[page,'source_group']),'mask_arm':mask_arm,'radius':radius,'status':'SKIPPED_ZERO_SEED','ambiguous_pixels':0});continue
        z=build_a0_a3(obs,t_seed,page)
        if not z['ambiguous'].any():
            observable_rows.append({'page':page,'source_group':str(meta.loc[page,'source_group']),'mask_arm':mask_arm,'radius':radius,'status':'SKIPPED_ZERO_AMBIGUOUS','authority_pixels':int(z['mask'].sum()),'ambiguous_pixels':0});del z;gc.collect();continue
        d.mkdir(parents=True,exist_ok=False)
        cv2.imwrite(str(d/'seed.png'),t_seed.astype(np.uint8)*255);cv2.imwrite(str(d/'authority.png'),z['mask'].astype(np.uint8)*255);cv2.imwrite(str(d/'ambiguous.png'),z['ambiguous'].astype(np.uint8)*255);cv2.imwrite(str(d/'ring2.png'),z['ring2'].astype(np.uint8)*255);np.save(d/'p_print.npy',z['p_print'])
        for arm,img in z['arms'].items():cv2.imwrite(str(d/f'{arm}.png'),img)
        rec={'page':page,'source_group':str(meta.loc[page,'source_group']),'mask_arm':mask_arm,'radius':radius,'status':'PROCESSED','seed_pixels':int(t_seed.sum()),'authority_pixels':int(z['mask'].sum()),'preserve_pixels':int(z['preserve'].sum()),'remove_pixels':int(z['remove'].sum()),'ambiguous_pixels':int(z['ambiguous'].sum()),'ring2_pixels':int(z['ring2'].sum()),'a3_gate_pixels':int(z['gate'].sum())}
        (d/'metadata.json').write_text(json.dumps(rec,indent=2));observable_rows.append(rec);print(page,mask_arm,'PROCESSED',rec['ambiguous_pixels']);del z;gc.collect()

observable=pd.DataFrame(observable_rows);observable.to_csv(RUN/'tables'/'observable_profile_summary.csv',index=False)
hash_rows=[]
for p in sorted((RUN/'observable').rglob('*')):
    if p.is_file():hash_rows.append({'path':str(p.relative_to(RUN)),'bytes':p.stat().st_size,'sha256':sha256_file(p)})
hashes=pd.DataFrame(hash_rows);hashes.to_csv(RUN/'audit'/'prediction_hashes.csv',index=False)
lock={'state':'CLOSED_BEFORE_CLEAN_TRUTH','created_utc':datetime.now(timezone.utc).isoformat(),'mask_arms':MASK_ARMS,'treatment_arms':ARMS,'fixed_radii':FIXED_RADII,'target_search_radii':TARGET_SEARCH_RADII,'hitl_target_profile':HITL_TARGET,'pprint_gate':PPRINT_GATE,'observable_rows':len(observable),'processed_rows':int((observable.status=='PROCESSED').sum()),'hash_table_sha256':hashlib.sha256(hashes.to_csv(index=False).encode()).hexdigest(),'operator_sha256':sha256_file(V89_OPERATOR_PATH)}
(RUN/'prediction_lock.json').write_text(json.dumps(lock,indent=2));print(json.dumps(lock,indent=2))


## 7. Post-lock truth scoring

Whole-authority outcomes are scored from the user's perspective: annotation improvement means movement materially closer to clean truth, and print safety records exact observed preservation, not-worsening and material worsening.


In [ ]:
require(json.loads((RUN/'prediction_lock.json').read_text())['state']=='CLOSED_BEFORE_CLEAN_TRUTH','Prediction lock not closed')

def safe_rate(n,d):return float(n/d) if d else np.nan

score=[];exclusions=[]
for page in paired_pages:
    obs=read_gray(obs_map[page]);clean=read_gray(clean_map[page]);clean_print=printed_ink(clean);clean_tol=cv2.dilate(clean_print.astype(np.uint8),np.ones((3,3),np.uint8))>0
    oe=np.abs(obs.astype(np.int16)-clean.astype(np.int16))
    for mask_arm in MASK_ARMS:
        d=RUN/'observable'/page/mask_arm
        if not complete_dir(d):
            q=observable[(observable.page==page)&(observable.mask_arm==mask_arm)]
            exclusions.append({'page':page,'mask_arm':mask_arm,'status':q.iloc[0].status if len(q) else 'MISSING','reason':'no complete experimental output'});continue
        authority=read_gray(d/'authority.png')>0;amb=read_gray(d/'ambiguous.png')>0
        truth_print=authority&clean_tol;truth_annotation=authority&(obs<DARK_THRESHOLD)&~clean_tol;truth_uncertain=authority&~truth_print&~truth_annotation
        for arm in ARMS:
            img=read_gray(d/f'{arm}.png');out_err=np.abs(img.astype(np.int16)-clean.astype(np.int16))
            ann_imp=truth_annotation&(out_err<=oe-IMPROVEMENT_MARGIN);print_pres=truth_print&(img==obs);print_not_worse=truth_print&(out_err<=oe);print_mat_worse=truth_print&(out_err>=oe+IMPROVEMENT_MARGIN)
            score.append({'page':page,'source_group':str(meta.loc[page,'source_group']),'mask_arm':mask_arm,'radius':json.loads((d/'metadata.json').read_text())['radius'],'arm':arm,'authority_pixels':int(authority.sum()),'ambiguous_pixels':int(amb.sum()),'truth_print_pixels':int(truth_print.sum()),'truth_annotation_pixels':int(truth_annotation.sum()),'truth_uncertain_pixels':int(truth_uncertain.sum()),'print_preserved_exact_observed':int(print_pres.sum()),'print_not_worsened':int(print_not_worse.sum()),'print_materially_worsened':int(print_mat_worse.sum()),'annotation_materially_improved':int(ann_imp.sum()),'print_exact_preservation_rate':safe_rate(print_pres.sum(),truth_print.sum()),'print_not_worsened_rate':safe_rate(print_not_worse.sum(),truth_print.sum()),'print_material_worsening_rate':safe_rate(print_mat_worse.sum(),truth_print.sum()),'annotation_material_improvement_rate':safe_rate(ann_imp.sum(),truth_annotation.sum())})
score=pd.DataFrame(score);score.to_csv(RUN/'tables'/'truth_scores_by_page_profile_arm.csv',index=False);pd.DataFrame(exclusions).to_csv(RUN/'tables'/'truth_scoring_exclusions.csv',index=False)
print('Scored rows:',len(score),'Excluded profile-pages:',len(exclusions))


## 8. Paired A3-versus-A0 deltas and page-clustered bootstrap

All deltas are signed so positive favours A3. The interaction at each broadened profile is:

`A3 benefit at Mk - A3 benefit at M0`.

A negative interaction means broadening erodes A3's benefit. A3 still transports if its direct benefit remains positive with a 95% lower confidence bound above zero.


In [ ]:
METRICS={
 'print_exact_preservation_rate':('Exact print preservation','A3-A0'),
 'print_not_worsened_rate':('Print not worsened','A3-A0'),
 'print_material_worsening_rate':('Reduction in materially worsened print','A0-A3'),
 'annotation_material_improvement_rate':('Annotation materially improved','A3-A0'),
}

def bootstrap(v,resamples=BOOTSTRAP_RESAMPLES,seed=SEED):
    v=np.asarray(v,float);v=v[np.isfinite(v)]
    if not len(v):return {'pages':0,'mean_delta':np.nan,'median_delta':np.nan,'ci95_low':np.nan,'ci95_high':np.nan,'wins':0,'losses':0,'ties':0}
    rng=np.random.default_rng(seed);means=np.array([rng.choice(v,len(v),replace=True).mean() for _ in range(resamples)])
    return {'pages':len(v),'mean_delta':v.mean(),'median_delta':np.median(v),'ci95_low':np.quantile(means,.025),'ci95_high':np.quantile(means,.975),'wins':int((v>0).sum()),'losses':int((v<0).sum()),'ties':int((v==0).sum())}

idx=['page','source_group','mask_arm','radius'];wide=score.pivot(index=idx,columns='arm',values=list(METRICS)).reset_index();flat=[]
for _,r in wide.iterrows():
    base={'page':r[('page','')],'source_group':r[('source_group','')],'mask_arm':r[('mask_arm','')],'radius':r[('radius','')]}
    for metric,(label,direction) in METRICS.items():
        a0=r[(metric,'A0_TELEA')];a3=r[(metric,'A3_PIXEL_GATE')];delta=(a3-a0) if direction=='A3-A0' else (a0-a3)
        flat.append({**base,'metric':metric,'label':label,'benefit_delta':delta})
deltas=pd.DataFrame(flat);deltas.to_csv(RUN/'tables'/'A3_vs_A0_page_deltas_by_mask_profile.csv',index=False)

summary=[]
for (mask_arm,metric),g in deltas.groupby(['mask_arm','metric']):
    q=bootstrap(g.benefit_delta,seed=SEED+sum(map(ord,mask_arm+metric)));summary.append({'mask_arm':mask_arm,'metric':metric,'label':g.label.iloc[0],**q})
summary=pd.DataFrame(summary)

m0=deltas[deltas.mask_arm=='M0'][['page','metric','benefit_delta']].rename(columns={'benefit_delta':'M0_delta'})
inter=deltas.merge(m0,on=['page','metric'],how='left');inter['transport_interaction']=inter.benefit_delta-inter.M0_delta
interaction=[]
for (mask_arm,metric),g in inter[inter.mask_arm!='M0'].groupby(['mask_arm','metric']):
    q=bootstrap(g.transport_interaction,seed=SEED+1000+sum(map(ord,mask_arm+metric)));interaction.append({'mask_arm':mask_arm,'metric':metric,'label':g.label.iloc[0],**{('interaction_'+k if k not in ['pages','wins','losses','ties'] else k):v for k,v in q.items()}})
interaction=pd.DataFrame(interaction)
summary.to_csv(RUN/'tables'/'A3_vs_A0_page_clustered_bootstrap_by_profile.csv',index=False);interaction.to_csv(RUN/'tables'/'A3_transport_interaction_bootstrap.csv',index=False)
display(summary.sort_values(['metric','mask_arm']));display(interaction.sort_values(['metric','mask_arm']))


## 9. Is 3 pixels sufficient?

The notebook answers this with explicit criteria rather than intuition:

1. **Morphology sufficiency:** M3 must reach or bracket the central reviewed-HITL range for the target morphology features.
2. **Transport sufficiency:** the experiment must include at least one profile at, and preferably beyond, the HITL target. Otherwise the tested grid has not stressed the production distribution.
3. **A3 robustness:** at the production-matched profile, A3's print-safety benefit must retain a positive page-clustered 95% lower bound.
4. **No hidden annotation trade:** the annotation-improvement delta is reported separately and never folded into a single utility.


In [ ]:
grid=pd.DataFrame(transformed_morph)
grid_med=grid.groupby('radius',as_index=False)[TARGET_FEATURES].median()
range_rows=[]
for _,r in grid_med.iterrows():
    inside=[]
    for f in TARGET_FEATURES:
        lo=float(hitl_m[f].quantile(.25));hi=float(hitl_m[f].quantile(.75));inside.append(lo<=r[f]<=hi)
    range_rows.append({'radius':int(r.radius),'features_inside_hitl_iqr':sum(inside),'all_features_inside_hitl_iqr':all(inside),**{f:r[f] for f in TARGET_FEATURES}})
range_df=pd.DataFrame(range_rows);range_df.to_csv(RUN/'tables'/'fixed_radius_match_to_hitl.csv',index=False)

s3=range_df[range_df.radius==3].iloc[0]
mt_print=summary[(summary.mask_arm=='MT')&(summary.metric=='print_material_worsening_rate')]
m3_print=summary[(summary.mask_arm=='M3')&(summary.metric=='print_material_worsening_rate')]
median_target=float(target.target_radius.median());p90_target=float(target.target_radius.quantile(.9));max_target=int(target.target_radius.max())

if bool(s3.all_features_inside_hitl_iqr) and p90_target<=3:
    dilation3='SUFFICIENT_TO_MATCH_CENTRAL_HITL_PROFILE'
elif median_target<=3<p90_target:
    dilation3='SUFFICIENT_FOR_TYPICAL_BUT_NOT_UPPER_TAIL_HITL_PROFILE'
else:
    dilation3='NOT_SUFFICIENT_TO_MATCH_HITL_PROFILE'

transport_status='NOT_ESTIMABLE'
if len(mt_print):
    transport_status='A3_BENEFIT_SURVIVES_MT' if float(mt_print.iloc[0].ci95_low)>0 else ('A3_POINT_ESTIMATE_POSITIVE_BUT_UNCERTAIN_MT' if float(mt_print.iloc[0].mean_delta)>0 else 'A3_BENEFIT_DOES_NOT_SURVIVE_MT')

verdict={'created_utc':datetime.now(timezone.utc).isoformat(),'notebook':'82_MASK_PROFILE_TRANSPORT','dilation_3_verdict':dilation3,'target_radius_median':median_target,'target_radius_p90':p90_target,'target_radius_max':max_target,'M3_features_inside_hitl_iqr':int(s3.features_inside_hitl_iqr),'target_features':TARGET_FEATURES,'A3_transport_at_MT':transport_status,'promotion_authority':False,'interpretation':'A fixed 3-pixel dilation is adequate only if it brackets reviewed HITL morphology; A3 robustness is judged separately by page-clustered confidence intervals at MT.'}
(RUN/'notebook82_verdict.json').write_text(json.dumps(verdict,indent=2));print(json.dumps(verdict,indent=2))


## 10. Visual diagnostics

These plots show whether the fixed-radius grid reaches the reviewed-HITL profile and whether A3's page-level benefit degrades as masks broaden.


In [ ]:
for f in TARGET_FEATURES:
    fig,ax=plt.subplots(figsize=(8,5));
    data=[];labels=[]
    data.append(paired_m[f].dropna());labels.append('Paired original')
    for r in FIXED_RADII:data.append(grid[grid.radius==r][f].dropna());labels.append(f'M{r}')
    data.append(hitl_m[f].dropna());labels.append('HITL reviewed')
    ax.boxplot(data,tick_labels=labels,showfliers=False);ax.set_title(f'Mask profile transport: {f}');ax.tick_params(axis='x',rotation=45);plt.tight_layout();plt.savefig(RUN/'visual'/f'morphology_{f}.png',dpi=160);plt.close(fig)

for metric,label in [(m,l[0]) for m,l in METRICS.items()]:
    q=summary[summary.metric==metric].copy();order=[f'M{r}' for r in FIXED_RADII]+['MT'];q['order']=q.mask_arm.map({x:i for i,x in enumerate(order)});q=q.sort_values('order')
    fig,ax=plt.subplots(figsize=(8,5));ax.errorbar(q.mask_arm,q.mean_delta,yerr=[q.mean_delta-q.ci95_low,q.ci95_high-q.mean_delta],fmt='o-',capsize=4);ax.axhline(0,color='black',linewidth=1);ax.set_ylabel('A3 benefit delta (positive favours A3)');ax.set_title(label);plt.tight_layout();plt.savefig(RUN/'visual'/f'transport_{metric}.png',dpi=160);plt.close(fig)
print('Visuals:',RUN/'visual')


## 11. Source-group stability and worst-page review queues


In [ ]:
source=[]
for (mask_arm,metric,group),g in deltas.groupby(['mask_arm','metric','source_group']):
    q=bootstrap(g.benefit_delta,resamples=5000,seed=SEED+sum(map(ord,mask_arm+metric+str(group))));source.append({'mask_arm':mask_arm,'metric':metric,'source_group':group,**q})
pd.DataFrame(source).to_csv(RUN/'tables'/'A3_vs_A0_by_profile_source_group.csv',index=False)

queues=[]
for mask_arm in MASK_ARMS:
    for metric in METRICS:
        q=deltas[(deltas.mask_arm==mask_arm)&(deltas.metric==metric)].sort_values('benefit_delta')
        queues.append(q.head(20).assign(queue='WORST_A3_DELTA'))
pd.concat(queues,ignore_index=True).to_csv(RUN/'tables'/'visual_review_queue_worst_A3_deltas.csv',index=False)


## 12. Final upload package

The package contains the verdict, principal tables, plots, exclusions and experiment identity. Heavy page-level images remain in the run directory and are referenced by the manifest.


In [ ]:
identity={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'notebook82-mask-profile-transport','development_only':True,'promotion_authority':False,'paired_pages':len(paired_pages),'hitl_reviewed_masks':len(hitl_reviewed),'hitl_draft_masks':len(hitl_drafts),'fixed_radii':FIXED_RADII,'target_search_radii':TARGET_SEARCH_RADII,'canonical_prepare_dilation':CANONICAL_PREP_DILATION,'pprint_gate':PPRINT_GATE,'bootstrap_resamples':BOOTSTRAP_RESAMPLES,'prediction_lock_sha256':sha256_file(RUN/'prediction_lock.json'),'operator_sha256':sha256_file(V89_OPERATOR_PATH),'software':{'python':sys.version,'opencv':cv2.__version__,'numpy':np.__version__,'pandas':pd.__version__,'platform':platform.platform()}}
(RUN/'experiment_identity.json').write_text(json.dumps(identity,indent=2,default=str))

summary_lines=['NOTEBOOK 82 - MASK-PROFILE TRANSPORT AND A3 ROBUSTNESS','',f'Paired pages: {len(paired_pages)}',f'Reviewed HITL masks: {len(hitl_reviewed)}',f'Fixed radii: {FIXED_RADII}',f'Target-search radii: {TARGET_SEARCH_RADII}','',f'3-pixel verdict: {verdict["dilation_3_verdict"]}',f'Target radius median / P90 / max: {median_target:.1f} / {p90_target:.1f} / {max_target}',f'A3 at target-matched profile: {transport_status}','','No production promotion is authorised.']
(RUN/'executive_summary.txt').write_text('\n'.join(summary_lines))

required=[RUN/'executive_summary.txt',RUN/'notebook82_verdict.json',RUN/'experiment_identity.json',RUN/'prediction_lock.json',RUN/'tables'/'paired_vs_hitl_morphology_shift.csv',RUN/'tables'/'target_matched_radius_by_page.csv',RUN/'tables'/'fixed_radius_match_to_hitl.csv',RUN/'tables'/'A3_vs_A0_page_clustered_bootstrap_by_profile.csv',RUN/'tables'/'A3_transport_interaction_bootstrap.csv',RUN/'tables'/'A3_vs_A0_by_profile_source_group.csv',RUN/'tables'/'truth_scoring_exclusions.csv']+sorted((RUN/'visual').glob('*.png'))
for p in required:
    require(p.exists(),f'Missing upload artefact: {p}');shutil.copy2(p,RUN/'upload'/p.name)
inv=[]
for p in sorted((RUN/'upload').iterdir()):
    if p.is_file():inv.append({'file':p.name,'bytes':p.stat().st_size,'sha256':sha256_file(p)})
pd.DataFrame(inv).to_csv(RUN/'upload'/'upload_manifest.csv',index=False)
shutil.make_archive(str(RUN/'notebook82_upload_bundle'),'zip',RUN/'upload')
print('\n'.join(summary_lines));print('Bundle:',RUN/'notebook82_upload_bundle.zip')
